## Preparing CSV files

In [ ]:
"""
import fsspec
FSSPEC_HTTPS = fsspec.filesystem("https")
def glob_url(url: str) -> list[str]:
    return FSSPEC_HTTPS.glob(url)
taxis_starred_url = "https://www.aviz.fr/nyc-taxi/yellow_*.csv.bz2"
glob_url(taxis_starred_url)
"""
import glob
import os
import ipywidgets as ipw
taxis_starred_url = f"{os.getenv('HOME')}/DATA_2026/yellow_*.csv.bz2"

USECOLS = [
'VendorID',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
]
DTYPE = {
 'VendorID': 'float64',
 'passenger_count': 'float64',
 'trip_distance': 'float64',
 'RatecodeID': 'float64',
 'PULocationID': 'int64',
 'DOLocationID': 'int64',
 'payment_type': 'int64',
 'fare_amount': 'float64',
 'extra': 'float64',
 'mta_tax': 'float64',
 'tip_amount': 'float64',
 'tolls_amount': 'float64',
 'improvement_surcharge': 'float64',
 'total_amount': 'float64',
}

def glob_url(url: str) -> list[str]:
    return glob.glob(url)

from progressivis.core.api import Module, asynchronize
    
def progress_bar(mod_, start_t) -> ipw.IntProgress:
    style = {'description_width': 'initial'}
    prog_wg = ipw.IntProgress(
        description="Total", min=0, max=1000, layout={"width": "100%"}, style=style
    )
    prog_crt = ipw.IntProgress(
        description="Crt:", min=0, max=1000, layout={"width": "100%"}, style=style
    )
    speed_w = ipw.IntText(description="Rows/sec:", value=0)
    files_w = ipw.Text(description="Processed files:", value="", style=style)    
    file_cnt = 0
    html_ = ipw.HTML()
    def _proc(m: Module, r: int = 0) -> None:
        nonlocal file_cnt
        val_, max_ = m.get_progress()
        prog_wg.value = val_
        if prog_wg.max != max_:
            prog_wg.max = max_
        valc_, maxc_ = m.get_progress_crt()
        prog_crt.value = valc_
        if prog_crt.max != maxc_:
            prog_crt.max = maxc_
        fname = m._last_opened.split("/")[-1]
        
        if fname != prog_crt.description:
            file_cnt += 1
            files_w.value = f"{file_cnt} / {m._n_files}"
            prog_crt.description = fname
        now_t = time.time()
        speed_w.value = len(mod_.result)//(now_t - start_t)

    async def aproc(m: Module, r: int) -> None:
        await asynchronize(_proc, m, r)
        
    mod_.on_after_run(aproc)
    return ipw.VBox([prog_crt, prog_wg, speed_w, files_w, html_])

## Loading many CSV files with ProgressiVis

In [ ]:
from progressivis import Sink, SimpleCSVLoader, PTable, Constant, Scheduler
from progressivis.core.api import Module, asynchronize
from progressivis.core import aio
import time

s = Scheduler.default
filenames = PTable(
                name="file_names",
                dshape="{filename: string}",
                data={"filename": glob_url(taxis_starred_url)},
)
cst = Constant(table=filenames, scheduler=s)
csv = SimpleCSVLoader(scheduler=s, usecols=USECOLS, dtype=DTYPE)
csv.input.filenames = cst.output.result
sink = Sink(scheduler=s)
sink.input.inp = csv.output.result
start_t = time.time()
bar = progress_bar(csv, start_t)
display(bar)
await s.start()
end_t = time.time()
delta_t = end_t - start_t
bar.children[-1].value = f"Elapsed time: {delta_t:.2f}"